# 🚀 Notebook do Professor (Demo) — Aula 06: Pipeline RAG completo

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 06/14 — Módulo 2: RAG · load → split → embed → retrieve → generate**  
**⏱️ 1h40min**  
**📄 PyMuPDF · RecursiveCharacterTextSplitter**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Construir um pipeline RAG completo funcional em ~50 linhas. O LLM responde sobre documentos reais que nunca estiveram no treinamento — com citação de página e trecho. Essa é a fundação do CKP02.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

### Slide 08 — Etapa 1 — carregar PDFs com PyMuPDF

In [ ]:
!pip install langchain langchain-community pymupdf langchain-ollama chromadb -q

from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

# Fazer upload do PDF no Colab
from google.colab import files
uploaded = files.upload()  # abre o seletor de arquivos
pdf_path = list(uploaded.keys())[0]

# Carregar — cada página vira um Document
loader = PyMuPDFLoader(pdf_path)
paginas = loader.load()

print(f"Páginas carregadas: {len(paginas)}")
print(f"Metadados da pág. 1: {paginas[0].metadata}")
# → {'source': 'manual.pdf', 'page': 0, 'total_pages': 24, ...}

print(f"Trecho da pág. 1:\n{paginas[0].page_content[:300]}")

# Carregar múltiplos PDFs de uma vez
pdfs = ["manual_1.pdf", "manual_2.pdf", "regulamento.pdf"]
todas_paginas = []
for pdf in pdfs:
    todas_paginas.extend(PyMuPDFLoader(pdf).load())
print(f"Total de páginas: {len(todas_paginas)}")

### Slide 09 — Etapa 2 — dividir em chunks com RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # ~600–1000 chars por chunk (boa prática para RAG)
    chunk_overlap=100,  # sobreposição para não quebrar frases no limite
    separators=["\n\n", "\n", ". ", " ", ""],  # ordem de preferência
)

# Dividir as páginas em chunks menores
chunks = splitter.split_documents(paginas)

print(f"Páginas originais: {len(paginas)}")
print(f"Chunks gerados:    {len(chunks)}")
print(f"Exemplo de chunk:\n{chunks[0].page_content}")
print(f"Metadados: {chunks[0].metadata}")
# → {'source': 'manual.pdf', 'page': 0} — página preservada!

# Ver distribuição de tamanhos dos chunks
tamanhos = [len(c.page_content) for c in chunks]
import statistics
print(f"Tamanho médio: {statistics.mean(tamanhos):.0f} chars")
print(f"Tamanho mín/máx: {min(tamanhos)} / {max(tamanhos)} chars")

### Slide 10 — Etapas 3 e 4 — embed e store no ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Etapas 3 + 4 em uma linha: embed todos os chunks e salva no ChromaDB
db = Chroma.from_documents(
    documents=chunks,                        # lista de chunks (Document)
    embedding=embeddings,                    # modelo de embedding
    persist_directory="/content/rag_ckp02",  # salva no disco
)
print(f"Chunks indexados: {db._collection.count()}")

# Retriever: interface de busca para a chain LCEL
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # recuperar os 3 chunks mais similares
)

# Testar o retriever isolado
docs_recuperados = retriever.invoke("qual é o prazo de entrega?")
for d in docs_recuperados:
    pg = d.metadata.get("page", "?")
    print(f"Pág. {pg}: {d.page_content[:120]}...")

### Slide 12 — O prompt de RAG — grounding e citação de fonte

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

prompt = ChatPromptTemplate.from_template(PROMPT_RAG)

### Slide 13 — Chain RAG completa — ~50 linhas, pipeline funcional

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

llm = ChatOllama(model="gpt-oss:120b", temperature=0)  # temp=0 para RAG (menos variação)

def formatar_contexto(docs) -> str:
    """Formata os docs recuperados com número de página para citação."""
    partes = []
    for i, doc in enumerate(docs, 1):
        fonte = doc.metadata.get("source", "doc")
        pg    = doc.metadata.get("page", "?")
        partes.append(f"[Trecho {i} — {fonte}, pág. {pg+1}]\n{doc.page_content}")
    return "\n\n---\n\n".join(partes)

# Chain LCEL completa — a magic pipe
chain_rag = (
    {
        "contexto":   retriever | RunnableLambda(formatar_contexto),
        "pergunta":   RunnablePassthrough(),  # pergunta passa direto
        "nome_doc":   RunnableLambda(lambda _: pdf_path),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Usar a chain
resposta = chain_rag.invoke("Qual é o prazo de entrega do produto?")
print(resposta)
# → "Conforme o documento manual.pdf, página 8, o prazo de entrega é..."

### Slide 17 — Pipeline completo em bloco único (~50 linhas)

In [ ]:
# ── 1. SETUP ────────────────────────────────────────────────
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# ── 2. INDEXAÇÃO (uma vez) ───────────────────────────────────
paginas  = PyMuPDFLoader("documento.pdf").load()
chunks   = RecursiveCharacterTextSplitter(800, 100).split_documents(paginas)
db       = Chroma.from_documents(chunks, OllamaEmbeddings(model="nomic-embed-text"),
                               persist_directory="/content/rag")
retriever = db.as_retriever(search_kwargs={"k":3})

# ── 3. CHAIN LCEL ───────────────────────────────────────────
def fmt(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','?')}, pág.{d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

chain = (
    {"contexto": retriever | RunnableLambda(fmt),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _:"documento.pdf")}
    | prompt | ChatOllama("gpt-oss:120b", temperature=0) | StrOutputParser()
)

# ── 4. USAR ─────────────────────────────────────────────────
print(chain.invoke("Qual é o prazo de garantia?"))

### Slide 18 — RunnablePassthrough e RunnableLambda — as peças de cola

In [ ]:
# O que acontece quando chain.invoke("Qual é o prazo?"):
#
# Input: "Qual é o prazo?"
#   ↓
# retriever.invoke("Qual é o prazo?") → [Doc1, Doc2, Doc3]
# formatar_contexto([Doc1, Doc2, Doc3]) → "[pág.8]\n...\n---\n[pág.12]\n..."
#   ↓
# contexto = "[pág.8]\n..."       ← resultado do retriever | lambda
# pergunta = "Qual é o prazo?"    ← RunnablePassthrough() — mesma string
# nome_doc = "documento.pdf"      ← RunnableLambda(lambda _: "...")
#   ↓
# prompt.format(contexto=..., pergunta=..., nome_doc=...) → mensagens
#   ↓
# llm(mensagens) → AIMessage
#   ↓
# StrOutputParser() → string final com citação de página

### Slide 22 — Python novo desta aula

In [ ]:
# 1. extend() — adicionar todos os itens de uma lista em outra
lista_a = [1, 2]
lista_a.extend([3, 4])  # [1, 2, 3, 4] — diferente de append([3, 4]) = [1, 2, [3, 4]]

# 2. enumerate() — iterar com índice
for i, doc in enumerate(docs, 1):  # começa em 1 — i=1,2,3...
    print(f"Trecho {i}: {doc.page_content}")

# 3. .get() em dict com valor padrão
pg = doc.metadata.get("page", "?")  # retorna "?" se "page" não existir

# 4. join() com separador multilinha
separador = "\n\n---\n\n"
texto     = separador.join(["Trecho A", "Trecho B"])

# 5. RunnablePassthrough e RunnableLambda — LCEL avançado
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

passthrough = RunnablePassthrough()        # x → x (identidade)
constante   = RunnableLambda(lambda _: "doc.pdf")  # ignora input, retorna constante
transformar = RunnableLambda(minha_funcao)    # qualquer função vira Runnable

# 6. statistics.mean() — média de uma lista
import statistics
media = statistics.mean([100, 200, 150])  # 150.0

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 06 · 2º Semestre**  
### Pipeline RAG completo sobre os PDFs do grupo ★★

*Grupo 3–4 · 20 minutos · Google Colab · PDFs do domínio obrigatórios*

1. Complete as 4 lacunas — caminhos dos PDFs, parâmetros do splitter, criação do vector store, montagem e invocação da chain.
2. Teste com 3 perguntas reais do domínio — perguntas que teriam respostas concretas nos documentos.
3. Compare sem vs. com RAG: faça a mesma pergunta ao ChatOllama direto e à chain_rag. Documente a diferença.
4. Verifique o grounding: faça uma pergunta que NÃO está nos documentos. A chain deve responder "Não encontrei essa informação" — não alucinar.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1:  os caminhos dos arquivos que acabaram de ser enviados pelo upload, usados para carregar cada PDF no loader.
>
> Lacuna 2:  o tamanho de cada chunk e a sobreposição entre eles — os mesmos valores usados na aula (800 e 100) — aplicados sobre a lista de páginas carregadas.
>
> Lacuna 3:  o nome do modelo de embedding do semestre, e os chunks e embeddings passados ao vector store na criação da coleção.
>
> Lacuna 4:  a pergunta repassada sem transformação para o prompt, e a pergunta real do domínio ao invocar a chain.

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb langchain-text-splitters -q

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import os
from google.colab import userdata, files

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# 👉 LACUNA 1: faça upload e carregue os PDFs do domínio
uploaded  = files.upload()
pdf_paths = [___]  # lista de caminhos dos PDFs enviados
paginas   = []
for p in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# 👉 LACUNA 2: crie o splitter e divida as páginas em chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=___,     # 800 é um bom ponto de partida
    chunk_overlap=___,  # 100 para não perder contexto nas bordas
)
chunks = splitter.split_documents(___)
print(f"Chunks: {len(chunks)}")

# 👉 LACUNA 3: crie o vector store com nomic-embed-text
embeddings = OllamaEmbeddings(model=___)
db = Chroma.from_documents(___, embedding=___, persist_directory="/content/ckp02")
retriever = db.as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 4: monte e invoque a chain RAG com grounding e citação
chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "nome_doc": RunnableLambda(lambda _: "PDFs do domínio")}
    | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print(chain_rag.invoke(___))  # sua pergunta sobre o domínio

## 📚 Referências da aula

- Paper Lewis, P. et al. — "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." NeurIPS, 2020. O paper original que cunhou o termo RAG. arxiv.org/abs/2005.11401
- Docs LangChain — RAG tutorial completo com PyMuPDF, Chroma e LCEL. python.langchain.com/docs/tutorials/rag
- Docs PyMuPDF — Documentação do loader LangChain com PyMuPDF. python.langchain.com/docs/integrations/document_loaders/pymupdf
- Docs RecursiveCharacterTextSplitter — Estratégias de chunking, parâmetros e separadores. python.langchain.com/docs/how_to/recursive_text_splitter
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings usados no RAG.

---

**→ Próxima Aula — Aula 07 · 21/09** — RAG avançado — chunking estratégico, reranking e RAGAS
  
Medir faithfulness e answer relevancy. Otimizar o pipeline. Entregar CKP02.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*